## 1. Setup & Why the Environment Actually Matters

You've already installed Python. The refresher part that actually matters for you *now*, three weeks
into a program that's about to hand you FastAPI, databases, and telecom OSS/BSS systems, is this:

- **Interpreter version** — `python3 --version`. A billing microservice written for Python 3.11 can
  silently misbehave on 3.9 (e.g. `match` statements, type-hint syntax). Version mismatches are a top-5
  cause of "works on my machine, breaks in the pipeline."
- **Virtual environments** — every real project isolates its dependencies (`venv`, `conda`, `poetry`).
  Two microservices on the same VM needing different versions of the same library is a daily reality in
  telecom OSS stacks where dozens of services run side by side.
- **`pip` / package management** — how a FastAPI project pulls in `fastapi`, `uvicorn`, `sqlalchemy`
  next week is the exact same mechanism you use today for anything else.
- **Where code actually runs** — a script you write today might run: on your laptop, inside a Docker
  container, as an Azure Function, or as a cron job on a Linux VM processing nightly CDR (Call Detail
  Record) files. Same Python, four different execution contexts.


**Where this shows up in the real stack:** Environment and dependency management is the first thing a
DevOps/CI pipeline checks before your code ever runs — it's literally step 1 of the CI/CD flow you
covered in Week 1. In telecom OSS/BSS, misaligned Python environments between dev, staging, and the
production billing engine is one of the most common causes of "it broke after deployment."


In [ ]:
# Live demo — run these one at a time and read the output

import sys
print("Python version:", sys.version)
print("Executable being used:", sys.executable)

# In a real project you'd see this reflected in a requirements.txt / pyproject.toml
# (we're not installing anything here — just showing what "the environment" means in practice)


## 2. Syntax, Indentation & Comments

Python has no `{ }` or `;`. **Indentation is the syntax.** This is the single most common source of
bugs for people coming from Java/C#/JS, and it is *exactly* the kind of bug that shows up in a FastAPI
route handler and silently changes which block of validation logic runs.

Key rules:
- A colon `:` opens a new indented block (`if`, `for`, `while`, `def`, `class`, ...).
- Indentation must be **consistent** within a block (4 spaces is the convention — don't mix tabs/spaces).
- `#` starts a comment. `"""triple quotes"""` are docstrings (also usable as multi-line comments).


**Where this shows up in the real stack:** Every FastAPI route, every Airflow DAG task, every AWS
Lambda handler you'll touch is just an indented Python block. Linters like `flake8`/`black` — which
get wired into CI pipelines — exist largely to catch indentation and style issues before they become
runtime bugs.


In [ ]:
# This block is fine — consistent indentation
network_status = "UP"
if network_status == "UP":
    print("Cell tower reporting normal service")
    print("No alert needed")
else:
    print("ALERT: tower down")

# Try breaking this on purpose: change the indentation of the second print()
# above so it's misaligned, run it, and read the error Python gives you.


## 3. Variables & Data Types

Python is **dynamically typed** (no need to declare a type) but **strongly typed** (it won't silently
turn `"5"` into `5` for you — you have to cast explicitly). The core primitive types you need cold:

| Type | Example | Telecom-flavored example |
|---|---|---|
| `int` | `42` | `call_duration_sec = 185` |
| `float` | `3.14` | `data_used_gb = 2.35` |
| `str` | `"hello"` | `msisdn = "9198xxxxxx21"` |
| `bool` | `True` | `is_roaming = False` |
| `NoneType` | `None` | `disconnect_reason = None` (call still active) |

**Why MSISDN (phone number) is a string, not an int:** leading zeros, `+` prefixes, and the fact that
you'll never do arithmetic on a phone number. This exact judgment call — "is this really a number, or
just digits?" — comes up constantly with subscriber IDs, IMSIs, and account numbers in real systems.

**Type casting**: `int("185")`, `float("2.35")`, `str(185)`, `bool(0)`. This matters enormously because
data coming from log files, CSVs, or API request bodies arrives as **strings**, even if it "looks like"
a number.


**Where this shows up in the real stack:** Every field in a Call Detail Record (CDR), every JSON field
in a FastAPI request body (`pydantic` models next week are essentially "typed variables with rules"),
every column in a billing database row — all of it starts as this exact primitive-type reasoning.
Get variables & types solid now, and next week's `pydantic` validation will feel like something you
already know with a new name on it.


In [ ]:
# Live demo — a mini subscriber record, field by field

msisdn = "9198xxxxxx21"       # str — subscriber's phone number
call_duration_sec = 185        # int — seconds
data_used_gb = 2.35             # float
is_roaming = False              # bool
plan_type = "postpaid"          # str
disconnect_reason = None        # NoneType — call is still active / not yet known

print(type(msisdn), type(call_duration_sec), type(data_used_gb), type(is_roaming), type(disconnect_reason))

# Casting: data arriving from a log file is ALWAYS text first
raw_log_field = "185"           # this came from a text log — it's a string!
call_duration_sec = int(raw_log_field)
print(call_duration_sec + 10, type(call_duration_sec))


### Your Turn (Core Task) — Parse a raw CDR line

A raw Call Detail Record line from a telecom switch looks like this (all fields arrive as **text**):

```
raw_cdr = "9876543210|247|3.8|True|prepaid"
```

Fields, in order: `msisdn | call_duration_sec | data_used_gb | is_roaming | plan_type`

1. Split `raw_cdr` on `"|"` into 5 separate string variables.
2. Cast each into its correct Python type (`str`, `int`, `float`, `bool`, `str`).
   — Note: casting the *string* `"True"` directly with `bool()` will NOT give you what you expect.
     Figure out why, and write a correct conversion.
3. Print each variable together with its `type()`.


In [2]:
# Your code here
raw_cdr = "9876543210|247|3.8|True|prepaid"
#split them by data type str,int,float,bool,str
msisdn, call_duration_sec, data_used_gb, is_roaming, plan_type = raw_cdr.split("|")
call_duration_sec = int(call_duration_sec)
data_used_gb = float(data_used_gb)
is_roaming = is_roaming == "True"  
print(msisdn,call_duration_sec,data_used_gb,is_roaming,plan_type)

9876543210 247 3.8 True prepaid


### Tricky Question

```python
a = "5"
b = 5
print(a == b)
print(a == str(b))
print(bool("False"))
print(bool(""))
```

Predict the output of all four lines **before** running it. Then explain in one sentence why
`bool("False")` behaves the way it does — this is the single most common "gotcha" when parsing
flags out of log files or CSV exports.


In [4]:
# Your answer / prediction here
a = "5"
b = 5
print(a == b)
print(int(a) == b)

False
True


## 4. Operators & Expressions

| Category | Operators | Example |
|---|---|---|
| Arithmetic | `+ - * / // % **` | `cost = minutes * rate` |
| Comparison | `== != > < >= <=` | `usage_gb > plan_limit_gb` |
| Logical | `and or not` | `is_active and not is_blacklisted` |
| Assignment | `= += -= *= /=` | `total_cost += call_cost` |
| Membership | `in`, `not in` | `"91" in msisdn` |

Two operators that trip almost everyone up at least once:
- `/` always returns a `float` (`7 / 2 = 3.5`), `//` is **floor division** (`7 // 2 = 3`).
- `%` is the **modulo** (remainder) — used constantly for "every Nth record" logic and billing cycles.

**Operator precedence** matters the same way it does in a spreadsheet formula — `and`/`or` chains
without parentheses are a classic source of billing-logic bugs.


**Where this shows up in the real stack:** This is the **rating engine** layer of a telecom BSS —
the component that takes raw usage (minutes, GB, SMS count) and applies pricing rules to produce a
bill line item. It's pure arithmetic + comparison + logical operators, at massive scale. The exact
same operators drive `WHERE` clause logic once you get to Database Integration next week.


In [ ]:
# Live demo — a tiny rating engine calculation

call_duration_sec = 247
rate_per_min = 0.85          # currency units per minute
plan_limit_gb = 5.0
data_used_gb = 3.8
is_roaming = False
is_blacklisted = False
is_active = True

minutes = call_duration_sec / 60          # true division -> float
call_cost = minutes * rate_per_min
print("Billable minutes:", round(minutes, 2), " Call cost:", round(call_cost, 2))

remaining_gb = plan_limit_gb - data_used_gb
print("Data remaining (GB):", round(remaining_gb, 2))
print("Over quota?", data_used_gb > plan_limit_gb)

can_bill = is_active and not is_blacklisted
print("Eligible to bill this subscriber?", can_bill)

# modulo in action — "flag every 10th record for a manual QA sample"
record_number = 40
print("Flag for QA sample?", record_number % 10 == 0)


### Your Turn (Core Task) — Roaming Surcharge Calculator

Given:
```python
base_cost = 120.0
is_roaming = True
is_international = False
loyalty_years = 4
```

Business rules:
1. If roaming, add a 35% surcharge to `base_cost`.
2. If international (and not roaming), add a flat 50 surcharge instead.
3. Regardless of the above, if `loyalty_years >= 3`, apply a 10% discount to the **final** total.

Write **one expression per rule** (you can use `if` only if you want to preview it — but this is meant
to be solvable with pure operators/expressions, no `if` needed yet). Print the final bill amount
rounded to 2 decimals.


In [ ]:
# Your code here


### Tricky Question

```python
print(10 / 3)
print(10 // 3)
print(-10 // 3)
print(10 % 3)
print(-10 % 3)
```

Predict every line. Pay special attention to the two negative-number lines — floor division and
modulo with negative numbers behave differently in Python than in some other languages (and this
has bitten real billing code that assumed C-style behaviour).


In [ ]:
# Your answer / prediction here


## 5. Control Flow — `if` / `elif` / `else`

```python
if condition:
    ...
elif other_condition:
    ...
else:
    ...
```

Rules that matter in practice:
- Conditions are evaluated **top to bottom**, and Python stops at the **first** `True` branch —
  order matters when conditions overlap.
- You can nest `if` inside `if`, but more than 2–3 levels deep is usually a sign the logic should be
  restructured (this is exactly what code review comments in a real repo will flag).
- `and` / `or` short-circuit: in `a and b`, if `a` is `False`, `b` is never evaluated. This is used
  deliberately in production code to avoid errors (e.g. checking `x is not None and x.value > 0`).


**Where this shows up in the real stack:** This *is* the business/validation logic layer. In a
FastAPI service (next week), every route handler is full of `if` statements validating a request
before it touches the database. In telecom OSS, network alerting systems use exactly this pattern to
classify severity: `if latency_ms > 150: critical elif latency_ms > 80: warning else: healthy`. In
BSS, call routing decides `if international: elif roaming: else: standard rate` before the rating
engine (Section 4) even runs its arithmetic.


In [ ]:
# Live demo — network health classification (this is a real NOC/NMS pattern)

latency_ms = 95
packet_loss_pct = 0.5

if latency_ms > 150 or packet_loss_pct > 5:
    status = "CRITICAL"
elif latency_ms > 80 or packet_loss_pct > 1:
    status = "WARNING"
else:
    status = "HEALTHY"

print(f"Latency={latency_ms}ms, Loss={packet_loss_pct}% -> Status: {status}")

# Call routing decision (BSS-style)
call_type = "roaming"   # "international" | "roaming" | "domestic"

if call_type == "international":
    rate_per_min = 12.0
elif call_type == "roaming":
    rate_per_min = 8.0
else:
    rate_per_min = 1.5

print("Applicable rate per minute:", rate_per_min)


### Your Turn (Core Task) — Fraud / Anomaly Flag

You're writing a lightweight fraud check that runs on every CDR before billing. Given:

```python
call_duration_sec = 340
call_cost = 0.0
is_international = True
subscriber_status = "active"     # "active" | "suspended" | "blacklisted"
```

Write `if/elif/else` logic that prints exactly one of:
- `"BLOCK: blacklisted subscriber"` — if `subscriber_status == "blacklisted"`
- `"HOLD: suspended subscriber"` — if `subscriber_status == "suspended"`
- `"FLAG: free international call anomaly"` — if the subscriber is active, the call is
  international, duration > 0, but cost is exactly 0 (classic "zero-rated leak" fraud pattern)
- `"OK: proceed to billing"` — otherwise

Test your logic against at least 2 different sets of input values (change the variables and re-run).


In [ ]:
# Your code here


### Tricky Question

```python
signal_strength_dbm = -85

if signal_strength_dbm > -70:
    print("Excellent")
if signal_strength_dbm > -85:
    print("Good")
if signal_strength_dbm > -100:
    print("Weak but connected")
```

1. How many lines print, and which ones? Why (careful — these are three separate `if` statements,
   not `if/elif/else`)?
2. Rewrite this correctly as a single `if/elif/else` chain so exactly ONE classification prints,
   and it correctly classifies -85 dBm.


In [ ]:
# Your answer / prediction here


## 6. Loops — `for`, `while`, `break`, `continue`

```python
for item in some_iterable:      # runs once per item — use when you know what you're iterating over
    ...

while condition:                 # runs until condition becomes False — use when you don't know
    ...                          # the count in advance (e.g. "keep retrying until success")
```

- `range(n)` → `0..n-1`. `range(a, b)` → `a..b-1`. `range(a, b, step)` for skipping.
- `break` exits the loop immediately. `continue` skips to the next iteration.
- **Infinite `while` loops are a real production risk** — always be able to point to the exact line
  that will eventually make the condition `False` (or use a max-retry counter as a safety net).

We haven't formally covered lists/dicts yet (that's next session) — for today, just know you can
write `for x in [1, 2, 3]:` to loop over a small hard-coded collection. We'll go deep on building and
manipulating these collections in the next class.


**Where this shows up in the real stack:** `for` loops are exactly how nightly batch jobs process a
folder of CDR files or a day's worth of network logs, record by record — the same `for` loop, just
looping over thousands of rows instead of 3. `while` + retry-counter is the standard pattern for
**resilient API calls** — e.g. a FastAPI service calling a downstream billing API that occasionally
times out, or a script polling an Azure job until it completes. You will write this *exact* pattern
again in a few weeks when you build retry logic around database/API calls.


In [ ]:
# Live demo — batch-processing a handful of call durations (a tiny stand-in for a CDR batch job)

call_durations_sec = [185, 0, 340, 47, 900, 12]   # (lists arrive properly next session — just an iterable for now)

total_cost = 0.0
rate_per_min = 0.85
flagged_zero_duration = 0

for duration in call_durations_sec:
    if duration == 0:
        flagged_zero_duration += 1
        continue                       # skip cost calc for a zero-duration record
    if duration > 600:
        print(f"Long call detected: {duration}s — flagging for manual review")
        # not using break here — we still want to bill it, just also flag it
    cost = (duration / 60) * rate_per_min
    total_cost += cost

print(f"Total billable cost: {round(total_cost, 2)}")
print(f"Zero-duration anomalies skipped: {flagged_zero_duration}")

# while + retry counter — the "resilient API call" pattern you'll reuse with FastAPI/DB calls
max_retries = 3
attempt = 0
call_succeeded = False

while attempt < max_retries and not call_succeeded:
    attempt += 1
    print(f"Attempt {attempt}: calling billing API...")
    call_succeeded = (attempt == 3)   # pretend it only succeeds on the 3rd try

print("Call succeeded!" if call_succeeded else "Gave up after max retries — escalate to on-call.")


### Your Turn (Core Task) — Nightly CDR Batch Summary

Given a batch of call durations (seconds):
```python
call_durations_sec = [45, 0, 620, 133, 0, 275, 910, 15]
```

Using a `for` loop, compute and print:
1. `total_calls` — total number of records.
2. `zero_duration_count` — how many are exactly 0 seconds (use `continue` to skip them from billing).
3. `long_call_count` — how many exceed 600 seconds (flag with a printed message per long call, but
   don't stop the loop — use the right tool, not `break`).
4. `total_cost` — using `rate_per_min = 0.85`, only for non-zero-duration calls.

Print all four results clearly labeled at the end.


In [ ]:
# Your code here


### Tricky Question

```python
attempt = 0
max_retries = 3

while attempt < max_retries:
    print(f"Trying connection... attempt {attempt}")
    attempt += 1
    if attempt == 2:
        break
else:
    print("All retries exhausted, connection failed.")

print("Done.")
```

1. What prints? Does the `else` block on the `while` loop execute? Why or why not — what does
   `while...else` actually mean in Python?
2. This is a real pattern: a `while...else` is used to distinguish "loop ended because condition
   became False" vs "loop ended because of `break`". Rewrite the retry logic from the hands-on demo
   using `while...else` so it prints `"Connected successfully"` only if the loop finished via
   `break` (success), and `"Escalate to on-call"` only if it ran out of retries naturally.


In [ ]:
# Your answer / prediction here


## Core-Path Assignment (Everyone — due before next session)

Build a small **"Mini Rating Engine"** script combining everything from today: variables & types,
type casting, operators, `if/elif/else`, and a `for` loop. No lists/dicts required beyond what
you've already used today (a plain list of numbers/strings to loop over is fine).

**Scenario:** You're given 5 raw CDR lines (as strings, pipe-separated, same format as the Section 3
task): `msisdn|call_duration_sec|is_international|is_roaming|subscriber_status`.

**Your script must, for each record:**
1. Parse and cast every field to its correct type.
2. Reject (print `"BLOCK"` + msisdn) if `subscriber_status` is `"blacklisted"`.
3. Otherwise compute a call cost:
   - Domestic (`not is_international and not is_roaming`): rate 1.5/min
   - Roaming: rate 8.0/min
   - International (not roaming): rate 12.0/min
4. Print a one-line summary per record: `msisdn`, applied rate type, and final cost (rounded to 2 dp).
5. At the end, print the **total revenue** across all non-blocked records.

Sample input to test with:
```python
raw_cdrs = [
    "9876543210|300|False|False|active",
    "9812345678|180|True|False|active",
    "9800011122|420|False|True|active",
    "9899988877|60|False|False|blacklisted",
    "9911122233|0|False|False|active",
]
```


In [ ]:
# Your assignment code here


## Competitive / Bonus Track (optional — for fast finishers)

These are intentionally harder and more open-ended. They exist so early finishers have something
real to chew on instead of sitting idle — nobody is required to complete these, and they don't affect
your core assignment grade. Bragging rights only.


### Bonus 1 — "FizzBuzz", Network-Ops Edition

Loop `for i in range(1, 31)` representing 30 consecutive minutes of a network health check. Print:
- `"PING"` if `i` is divisible by 3 (a routine ping check happens)
- `"SNMP POLL"` if `i` is divisible by 5 (an SNMP poll happens)
- `"PING SNMP POLL"` if divisible by both (in that exact order)
- otherwise just the minute number

This is the classic FizzBuzz interview question — you will very likely be asked a variant of this in
a technical screen.


In [ ]:
# Your code here


### Bonus 2 — SLA Breach Streak Detector

Given a list of latency readings taken once per minute:
```python
latency_log_ms = [45, 60, 190, 210, 205, 55, 300, 310, 320, 315, 40]
```
An SLA breach is any reading `> 150`. Using a `while` or `for` loop (your choice), find and print the
**longest consecutive streak of breaches** and the minute-index range it occurred in (0-indexed).
Expected answer for the sample data: longest streak is 4, from index 6 to 9.


In [ ]:
# Your code here


### Bonus 3 — Interview-Style Debugging Challenge

This snippet is meant to classify subscribers into billing tiers, but it has **three separate bugs** —
a mix of a type bug, an operator/precedence bug, and a control-flow ordering bug. Find and fix all
three without rewriting it from scratch.

```python
usage_gb = "12.5"
loyalty_years = 2
is_vip = True

if usage_gb > 10 and loyalty_years >= 3 or is_vip:
    tier = "premium discount"
elif usage_gb > 10:
    tier = "standard"
else:
    tier = "premium discount"
    tier = "light user"

print(tier)
```


In [ ]:
# Fix and paste the corrected snippet here
